# Hand Gesture Lab: Landmarks & Rule-Based Classification

**Fontys ICT · Educational notebook**

In this notebook you will:

1. Open your webcam and capture a frame.
2. Detect the 21 hand landmarks provided by MediaPipe Hands.
3. See how those landmarks can be normalised into an optional 63-number feature vector (the rule classifier itself works on raw landmarks).
4. Build a 63-number feature vector.
5. Watch the rule-based classifier decide between **open**, **fist**, and **custom**.
6. Experiment with the confidence threshold.
7. Discuss failure cases.

> **Note:** This notebook needs a local webcam and a display. It does **not** run on Google Colab.

In [ ]:
import sys
import time

import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt

from gesture_lab.gesture_lib import (
    LANDMARK_NAMES,
    RuleClassifier,
    feature_vector,
    landmarks_to_array,
    normalize_landmarks,
)

print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("NumPy:", np.__version__)
print("Python:", sys.version)


## 1. Capture one frame from the webcam

`cv2.VideoCapture(0)` usually opens the built-in camera on macOS and Windows. If it fails, try `1`.

In [ ]:
cap = cv2.VideoCapture(0)
ok, frame = cap.read()
cap.release()
assert ok, "Could not read a frame from the camera. Try a different camera index."

print("Frame shape:", frame.shape)  # H x W x 3, BGR order
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Captured frame")
plt.show()


## 2. Detect the 21 hand landmarks

MediaPipe Hands returns 21 landmarks. Each landmark has `(x, y, z)` coordinates normalised to the image size, so values are between 0 and 1.

In [ ]:
hands = mp.solutions.hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)

rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
result = hands.process(rgb)

if result.multi_hand_landmarks:
    landmarks = result.multi_hand_landmarks[0]
    annotated = frame.copy()
    mp.solutions.drawing_utils.draw_landmarks(
        annotated,
        landmarks,
        mp.solutions.hands.HAND_CONNECTIONS,
    )
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("21 MediaPipe landmarks")
    plt.show()
else:
    print("No hand detected. Try moving your hand closer to the camera.")


## 3. Explore the landmark indices

The landmarks are returned in a fixed order. Here are the names and their indices.

In [ ]:
for i, name in enumerate(LANDMARK_NAMES):
    print(f"{i:2d}: {name}")


In [ ]:
# Overlay landmark numbers on the frame
annotated = frame.copy()
arr = landmarks_to_array(landmarks)
height, width = frame.shape[:2]
for i, (x, y, z) in enumerate(arr):
    cx, cy = int(x * width), int(y * height)
    cv2.circle(annotated, (cx, cy), 5, (0, 255, 0), -1)
    cv2.putText(annotated, str(i), (cx + 6, cy - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Landmark indices (0 is the wrist)")
plt.show()


## 4. Normalise the landmarks

Two important preprocessing steps:

1. **Translate** the wrist (landmark 0) to the origin.
2. **Scale** by hand size.

This makes the representation much less sensitive to where the hand is in the frame or how far it is from the camera. It does *not* remove rotation or differences between people.

In [ ]:
original = landmarks_to_array(landmarks)
normalised = normalize_landmarks(landmarks)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(original[:, 0], original[:, 1], c="blue")
axes[0].scatter(original[0, 0], original[0, 1], c="red", s=100, label="wrist")
axes[0].invert_yaxis()
axes[0].set_aspect("equal")
axes[0].set_title("Original landmarks")
axes[0].legend()

axes[1].scatter(normalised[:, 0], normalised[:, 1], c="green")
axes[1].scatter(normalised[0, 0], normalised[0, 1], c="red", s=100, label="wrist")
axes[1].invert_yaxis()
axes[1].set_aspect("equal")
axes[1].set_title("Normalised landmarks")
axes[1].legend()

plt.tight_layout()
plt.show()


## 5. Build the feature vector

The optional `feature_vector` helper flattens the normalised landmarks into 21 × 3 = 63 numbers.  The rule classifier in this notebook works directly on the raw landmarks; the feature vector is there if you want to feed the data into a different model later.


In [ ]:
vec = feature_vector(landmarks)
print("Feature vector shape:", vec.shape)
print("First 12 values:", vec[:12])
print("Min:", vec.min(), "Max:", vec.max())


## 6. Use the rule-based classifier

The classifier is *not* a machine-learning model. It is a short set of human-written rules:

- Count how many of the four long fingers are extended.
- If 3 or 4 are extended → **open**.
- If 0 are extended → **fist**.
- Otherwise → **custom**.

A finger is considered extended when its tip is higher (smaller y-value) than its PIP joint.

In [ ]:
classifier = RuleClassifier()
label, score = classifier.classify(landmarks)
print(f"Detected: {label} (score {score:.2f})")

# Show which fingers are extended
arr = landmarks_to_array(landmarks)
finger_names = ["index", "middle", "ring", "pinky"]
pairs = [(8, 6), (12, 10), (16, 14), (20, 18)]
for name, (tip, pip) in zip(finger_names, pairs):
    extended = arr[tip, 1] < arr[pip, 1] - 0.025
    print(f"  {name:6s}: tip_y={arr[tip,1]:.3f}  pip_y={arr[pip,1]:.3f}  extended={extended}")


## 7. Effect of the confidence threshold

The classifier returns a score, but the game only jumps when the score is above a threshold. Try a few thresholds and see when a jump would fire.

In [ ]:
for threshold in [0.3, 0.5, 0.7, 0.9]:
    fires = score >= threshold
    print(f"Threshold {threshold:.1f}: score {score:.2f} -> jump fires? {fires}")


## 8. Optional live demo

Run the cell below to classify your hand in real time for 10 seconds. It updates the displayed frame inside the notebook.

In [ ]:
from IPython.display import clear_output

threshold = 0.6
cap = cv2.VideoCapture(0)
start = time.time()

while time.time() - start < 10:
    ok, frame = cap.read()
    if not ok:
        break
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res = hands.process(rgb)
    label = "none"
    score = 0.0
    if res.multi_hand_landmarks:
        lm = res.multi_hand_landmarks[0]
        label, score = classifier.classify(lm)
        mp.solutions.drawing_utils.draw_landmarks(
            rgb, lm, mp.solutions.hands.HAND_CONNECTIONS
        )
    text = f"{label} ({score:.2f})"
    cv2.putText(rgb, text, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    plt.figure(figsize=(8, 6))
    plt.imshow(rgb)
    plt.axis("off")
    plt.show()
    clear_output(wait=True)

cap.release()


## 9. Failure cases and reflection

No classifier is perfect. Try each test and record what happens.

1. **Rotate your hand** so the palm faces sideways. Does the rule classifier still work?
2. **Move far from the camera** so your hand becomes small. Does it still detect the gesture?
3. **Change the lighting** — very bright backlight or a dark room.
4. **Ask a different person** to try the same gestures.
5. **Why does the score not guarantee correctness?**

### Questions

- What did normalisation change? Why was it useful?
- Why does the classifier ignore the thumb?
- When would a higher threshold help? When would it hurt?